# Human In the Loop Middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [2]:
agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    tools = [read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
            "send_email_tool":{"allowed_decissions":["approve","edit","reject"]
            },
            "read_email_tool":False,
            }
        )
    ]
)

In [3]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import content
from langchain_core.runnables import config
config = {"configurable": {"thread_id": "test-approve"}}
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [4]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='bbac64c1-3dc7-417c-bdcb-1db0272b4d73'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let's see. The user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. I need to check which tool to use here. The available tools are read_email_tool and send_email_tool. Since the action is sending, it's definitely the send_email_tool.\n\nLooking at the parameters required for send_email_tool: recipient, subject, and body. The user provided all three: recipient is john@test.com, subject is 'Hello', and body is 'How are you?'. I just need to structure these into the JSON object as arguments. Make sure the keys are correct and the values are strings. No missing parameters, so the tool call should be straightforward. Let me double-check the required fields again to confirm. Yep, all 

In [5]:
from langgraph.types import Command

# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

print(f"✅ Result: {result['messages'][-1].content}")

✅ Result: The email has been successfully sent to john@test.com with the subject "Hello". Let me know if you need anything else!
